In [0]:
 %pip install seaborn
 %restart_python

In [0]:
# ============================================
# Setup global
# ============================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import os, warnings
warnings.filterwarnings("ignore")

# --- Palette "cinéma premium" ---
PALETTE = ["#C9A227", "#B23A48", "#1B2A41", "#6B8F71", "#DDA15E", "#4A5F8A"]
sns.set_theme(style="whitegrid", palette=PALETTE, context="notebook")
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.titleweight": "bold",
    "axes.titlesize": 15,
    "axes.labelsize": 12,
    "axes.edgecolor": "#888",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
})

# Dossier de sortie 
os.makedirs("assets", exist_ok=True)

# --- Chargement des tables Gold ---
GOLD = {
    "top_box":    spark.table("workspace.default.gold_top_boxoffice"),
    "scatter":    spark.table("workspace.default.gold_budget_vs_revenue"),
    "genres":     spark.table("workspace.default.gold_genres_by_decade"),
    "yearly":     spark.table("workspace.default.gold_yearly_stats"),
    "top_rated":  spark.table("workspace.default.gold_top_rated"),
    "corr":       spark.table("workspace.default.gold_correlation"),
}
# Exception documentée : distribution des recettes = besoin de la Silver brute
df_silver = spark.table("workspace.default.films_silver")

# --- Vérification ---
for name, t in GOLD.items():
    print(f"✅ {name:10s} → {t.count():>8,} lignes")
print(f"\n✅ Silver     → {df_silver.count():>8,} lignes")
print(f"\n📁 Sortie   → {os.path.abspath('assets')}/")

In [0]:
# ============================================
# Top 15 du box-office
# ============================================
top = GOLD["top_box"].toPandas().sort_values("revenue")
top["revenue_M"] = top["revenue"] / 1e6

fig, ax = plt.subplots(figsize=(13, 7))
colors = sns.color_palette("YlOrRd", n_colors=len(top))[::-1]
bars = ax.barh(top["title"].str.slice(0, 45), top["revenue_M"],
               color=colors, edgecolor="white", linewidth=1)

for bar, val in zip(bars, top["revenue_M"]):
    ax.text(val + 10, bar.get_y() + bar.get_height()/2,
            f"${val:,.0f}M", va="center", fontsize=10, fontweight="bold")

ax.set_xlabel("Recettes au box-office (millions USD)")
ax.set_title("🎬 Les 15 plus gros succès du box-office mondial",
             fontsize=17, pad=20, loc="left")
ax.text(0, 1.02, "Source : dataset IMDb × TMDB  ·  couche Gold",
        transform=ax.transAxes, fontsize=10, color="#666", style="italic")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"${x:,.0f}M"))
sns.despine(left=True)
plt.savefig("assets/01_top_boxoffice.png")
plt.show()

# Insight
lead = top.iloc[-1]
print(f"💡 {lead['title']} domine avec ${lead['revenue_M']:,.0f}M de recettes")

In [0]:
# ============================================
# Budget vs Recettes (échelle log)
# ============================================
s = GOLD["scatter"].toPandas().dropna(subset=["budget", "revenue"])
s = s[(s["budget"] > 0) & (s["revenue"] > 0)].copy()
s["roi"] = s["revenue"] / s["budget"]

fig, ax = plt.subplots(figsize=(12, 8))
sc = ax.scatter(s["budget"]/1e6, s["revenue"]/1e6,
                c=np.log10(s["roi"]), cmap="RdYlGn",
                s=45, alpha=0.7, edgecolors="white", linewidths=0.4,
                vmin=-1, vmax=1)

max_val = max(s["budget"].max(), s["revenue"].max()) / 1e6
ax.plot([0.1, max_val], [0.1, max_val], "--", color="#555", lw=1.3,
        label="Seuil de rentabilité (Recettes = Budget)")

for _, row in s.nlargest(3, "roi").iterrows():
    ax.annotate(row["title"][:25],
                xy=(row["budget"]/1e6, row["revenue"]/1e6),
                xytext=(row["budget"]/1e6 * 1.8, row["revenue"]/1e6 * 1.3),
                fontsize=9, fontweight="bold", color="#1B2A41",
                arrowprops=dict(arrowstyle="->", color="#333", lw=0.8))

ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Budget (millions USD, échelle log)")
ax.set_ylabel("Recettes (millions USD, échelle log)")
ax.set_title("ROI cinéma : budget vs recettes",
             fontsize=17, pad=20, loc="left")
ax.text(0, 1.02, f"n = {len(s):,} films  ·  couleur = ROI (vert = très rentable)",
        transform=ax.transAxes, fontsize=10, color="#666", style="italic")
cbar = plt.colorbar(sc, ax=ax, shrink=0.8)
cbar.set_label("log10(Recettes / Budget)")
ax.legend(loc="upper left", frameon=True)
plt.savefig("assets/02_budget_vs_revenue.png")
plt.show()

print(f"ROI médian : {s['roi'].median():.2f}×  ·  meilleur : {s['roi'].max():.1f}×")

In [0]:
# ============================================
# Genres par décennie (stacked area)
# ============================================
gen = GOLD["genres"].toPandas()
# Exclure le genre 'Unknown' (métadonnée manquante, ~211k films — fausse la lecture)
gen = gen[gen["genre"] != "Unknown"]
# Exclure les décennies aberrantes (dates erronées : ~1800–1880, 2030–2090)
gen = gen[(gen["decennie"] >= 1890) & (gen["decennie"] <= 2030)]
top5 = gen.groupby("genre")["count"].sum().nlargest(5).index
pivot = (gen[gen["genre"].isin(top5)]
         .pivot(index="decennie", columns="genre", values="count")
         .fillna(0).sort_index())

fig, ax = plt.subplots(figsize=(13, 6))
pivot.plot.area(ax=ax, color=PALETTE[:5], alpha=0.88, linewidth=1.5)
ax.set_xlabel("Décennie")
ax.set_ylabel("Nombre de films produits")
ax.set_title("🎭 Évolution de la production par genre",
             fontsize=17, pad=20, loc="left")
ax.text(0, 1.02, "Top 5 genres les plus représentés dans le dataset",
        transform=ax.transAxes, fontsize=10, color="#666", style="italic")
ax.legend(loc="upper left", frameon=False, ncol=2)
sns.despine()
plt.savefig("assets/03_genres_decennies.png")
plt.show()

# 💡 Insight : genre qui a le plus progressé
growth = (pivot.iloc[-1] - pivot.iloc[0]).sort_values(ascending=False)
print(f"💡 Genre en plus forte croissance : {growth.index[0]} "
      f"(+{int(growth.iloc[0])} films)")

In [0]:
# ============================================
#Matrice de corrélation (triangle)
# ============================================
corr = GOLD["corr"].toPandas().corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(9, 7))
cmap = sns.diverging_palette(230, 20, as_cmap=True)
sns.heatmap(corr, mask=mask, cmap=cmap, vmin=-1, vmax=1, center=0,
            annot=True, fmt=".2f", square=True, linewidths=.5,
            cbar_kws={"shrink": .7}, ax=ax,
            annot_kws={"size": 11, "weight": "bold"})
ax.set_title("Corrélations entre variables clés",
             fontsize=16, pad=20, loc="left")
ax.text(0, 1.02, "Corrélations de Pearson  ·  triangle inférieur uniquement",
        transform=ax.transAxes, fontsize=10, color="#666", style="italic")
plt.savefig("assets/04_correlation_heatmap.png")
plt.show()

# 💡 Insight : corrélation la plus forte avec revenue
if "revenue" in corr.columns:
    strongest = corr["revenue"].drop("revenue").abs().idxmax()
    print(f"💡 Variable la plus corrélée aux recettes : {strongest} "
          f"(r = {corr.loc[strongest, 'revenue']:.2f})")

In [0]:
# ============================================
# DASHBOARD 2x2 — synthèse visuelle
# ============================================
yearly   = GOLD["yearly"].toPandas().sort_values("annee")
rated    = GOLD["top_rated"].toPandas().sort_values("vote_average")
revenues = (df_silver.select("revenue")
              .filter("revenue > 100000")
              .sample(fraction=0.15, seed=42)
              .toPandas()["revenue"] / 1e6)

fig = plt.figure(figsize=(18, 12))
gs  = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.25)

# (1) Note moyenne par année
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(yearly["annee"], yearly["note_moy"], color=PALETTE[1], lw=2.5)
ax1.fill_between(yearly["annee"], yearly["note_moy"], alpha=0.2, color=PALETTE[1])
ax1.set_title("Note IMDb moyenne par année", fontsize=13, loc="left")
ax1.set_ylabel("Note moyenne /10")
ax1.set_ylim(4.5, 8.5)

# (2) Volume de production par année
ax2 = fig.add_subplot(gs[0, 1])
ax2.bar(yearly["annee"], yearly["nb_films"],
        color=PALETTE[0], edgecolor="white")
ax2.set_title("Volume de films produits par an", fontsize=13, loc="left")
ax2.set_ylabel("Nombre de films")

# (3) Top 10 films par note
ax3 = fig.add_subplot(gs[1, 0])
bars = ax3.barh(rated["title"].str.slice(0, 30), rated["vote_average"],
                color=PALETTE[2])
for b, v in zip(bars, rated["vote_average"]):
    ax3.text(v - 0.15, b.get_y() + b.get_height()/2, f"{v:.1f}",
             va="center", ha="right", color="white",
             fontweight="bold", fontsize=10)
ax3.set_xlim(0, 10)
ax3.set_title("Top 10 films par note (min. 5k votes)", fontsize=13, loc="left")

# (4) Distribution des recettes (log)
ax4 = fig.add_subplot(gs[1, 1])
sns.histplot(revenues, bins=50, ax=ax4, color=PALETTE[3],
             kde=True, log_scale=(True, False))
ax4.set_title("Distribution des recettes (log)", fontsize=13, loc="left")
ax4.set_xlabel("Recettes (M USD, log)")
ax4.set_ylabel("Fréquence")

fig.suptitle("Dashboard — Analyse du cinéma mondial (IMDb × TMDB)",
             fontsize=19, fontweight="bold", y=0.98)
fig.text(0.5, 0.955, "4 insights clés extraits du dataset",
         ha="center", fontsize=11, color="#666", style="italic")
plt.savefig("assets/05_dashboard.png")
plt.show()

print("✅ Dashboard généré — assets/05_dashboard.png")

In [0]:
# ============================================
# Vérification & résumé
# ============================================
files = sorted(os.listdir("assets"))
print("📁 Fichiers générés dans assets/ :\n")
for f in files:
    size = os.path.getsize(f"assets/{f}") / 1024
    print(f"  ✓ {f:<35} {size:>6.1f} KB")
print(f"\n✅ {len(files)} visuels prêts à être committés sur GitHub")